# Policies under risk, drift, and memory

Risk changes which backup is attractive. Drift changes the backup's target.
Hidden memory determines whether that target can be represented at all.

This notebook turns the three follow-ups from notebook 02 into one connected
investigation. Act I compares SARSA, Expected SARSA, Q-learning, and Double
Q-learning on a risky shortcut. Act II aligns their TD errors around a
repeated within-episode reliability shock. Act III holds average wall
frequency fixed while increasing wall persistence, then compares a
position-only representation with one that includes the wall bit.

This is a controlled comparison of learning mechanics, not a universally
tuned leaderboard. Every method receives the same transition budget,
exploration schedule, and root seeds. Held-out greedy evaluation is the
primary performance view; exploratory training return answers a different
question.

QUICK mode runs every act with a small seed panel. Set QUICK to False only
for a deliberate research run. Each act can also reopen an existing
Protocol-v2 directory, so rerunning plots never requires retraining.


In [ ]:
from __future__ import annotations

from dataclasses import replace
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rllab.environments import MarkovWall, NonstationarityConfig
from rllab.evaluation import (
    episodes_to_threshold,
    evaluation_checkpoint_summary,
)
from rllab.experiments import (
    AgentSpec,
    ArtifactSpec,
    EnvironmentSpec,
    EvaluationScenario,
    ExecutionSpec,
    Experiment,
    ExperimentConfig,
    ExperimentResult,
    PolicyEvaluationSpec,
    RunStore,
    StepRetentionSpec,
    estimate_run,
    make_environment,
)
from rllab.metrics import paired_seed_contrast
from rllab.theory import value_iteration
from rllab.visualization import (
    plot_final_distribution,
    plot_learning_curves,
    plot_maze,
    plot_paired_contrasts,
    plot_policy,
    plot_transition_noise,
)

SMOKE = os.environ.get("RL_LAB_NOTEBOOK_SMOKE") == "1"
QUICK = True
SHOW_PROGRESS = False  # avoids stale Jupyter widget models; the CLI has live progress
AGENT_ORDER = ("q_learning", "sarsa", "expected_sarsa", "double_q_learning")
AGENT_LABELS = {
    "q_learning": "Q-learning",
    "sarsa": "SARSA",
    "expected_sarsa": "Expected SARSA",
    "double_q_learning": "Double Q-learning",
}
EXISTING_RUNS: dict[str, str | Path | None] = {
    "risk": None,
    "drift": None,
    "memory": None,
}

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "rllab").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the rl-lab repository.")

REPO_ROOT = find_repo_root(Path.cwd())
configured_results = os.environ.get("RL_LAB_NOTEBOOK_RESULTS")
RESULTS_DIR = Path(configured_results) if configured_results else REPO_ROOT / "results"
N_RESAMPLES = 100 if SMOKE else (400 if QUICK else 2_000)
POLICY_SEED = 0

available_styles = set(plt.style.available)
plot_style = next(
    (
        style
        for style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot")
        if style in available_styles
    ),
    "default",
)
plt.style.use(plot_style)

def add_method(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    if "agent" in result:
        result["method"] = result["agent"].map(AGENT_LABELS).fillna(result["agent"])
    return result

def last_checkpoint(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.loc[
        frame["checkpoint_episode"].eq(
            frame.groupby("trial_id")["checkpoint_episode"].transform("max")
        )
    ].copy()

def run_or_reopen(label: str, config: ExperimentConfig) -> ExperimentResult:
    display(pd.Series(estimate_run(config).as_dict(), name=f"{label} preflight"))
    existing = EXISTING_RUNS[label]
    if existing is not None:
        path = Path(existing)
        path = path if path.is_absolute() else REPO_ROOT / path
        store = RunStore.open(path)
        if store.manifest.experiment_name != config.name:
            raise ValueError(
                f"{label} run is {store.manifest.experiment_name!r}, expected {config.name!r}"
            )
        print(f"Reopened {label}: {store.run_directory}")
        return ExperimentResult(
            experiment_id=store.manifest.run_id,
            run_directory=store.run_directory,
            metadata={"reopened": True},
        )
    print(f"Starting {label} run with {len(config.trials())} matched trials...")
    result = Experiment(config).run(persist=True, progress=SHOW_PROGRESS)
    print(f"Completed {label}: {result.run_directory}")
    return result

def final_q_tables(result: ExperimentResult) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    final_rows = (
        result.snapshots.query("episode >= 0")
        .sort_values(["trial_id", "episode"])
        .groupby("trial_id", as_index=False)
        .tail(1)
        .copy()
    )
    tables = {}
    for row in final_rows.itertuples(index=False):
        snapshots = result.q_snapshots(
            row.trial_id,
            keys=(row.snapshot_key,),
        )
        tables[row.trial_id] = snapshots[row.snapshot_key]
    return final_rows, tables

def modal_policy(q_tables: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    policies = np.stack([np.argmax(table, axis=1) for table in q_tables])
    modes = np.array(
        [
            np.bincount(policies[:, state], minlength=q_tables[0].shape[1]).argmax()
            for state in range(policies.shape[1])
        ],
        dtype=int,
    )
    consensus = np.mean(policies == modes[None, :], axis=0)
    return modes, consensus


## The four backups and the hypotheses

The algorithms differ at one deliberately narrow point: the bootstrap in
the one-step target.

| Method | Bootstrap at the next state |
|---|---|
| SARSA | the sampled exploratory action |
| Expected SARSA | the expectation under the exploratory policy |
| Q-learning | the largest current action value |
| Double Q-learning | one table selects and the other evaluates |

We keep alpha, gamma, epsilon, seeds, and episode budgets fixed. Equal
hyperparameters are useful for isolating mechanisms, but they do not imply
that every method is individually tuned to its best setting.


In [ ]:
hypotheses = pd.DataFrame(
    [
        {
            "act": "risk",
            "prediction": "SARSA-style targets price exploratory accidents into the learned route.",
            "primary evidence": "paired held-out return and learned policy",
        },
        {
            "act": "risk",
            "prediction": "Expected SARSA removes next-action sampling noise from SARSA.",
            "primary evidence": "late TD-error variance",
        },
        {
            "act": "drift",
            "prediction": "every fixed-step-size learner shows a TD shock when reliability falls.",
            "primary evidence": "event-aligned absolute TD error",
        },
        {
            "act": "memory",
            "prediction": "backup choice cannot recover a wall bit omitted from the observation.",
            "primary evidence": "position-only versus wall-aware held-out return",
        },
    ]
)
display(hypotheses)


## Act I — Risk: shortcut or detour?

The first act reuses the versioned heterogeneous-route experiment. The
central route is short but locally unreliable; the detour is longer. This
stationary, fully observed environment has an exact finite MDP, so return,
value error, and tie-aware policy error are all legitimate diagnostics.

Training return includes epsilon-greedy actions and updates. Held-out return
freezes a clone of the learned policy and takes greedy actions on a seed
panel disjoint from training. We treat the training seed—not an evaluation
episode—as the independent unit.


In [ ]:
full_risk_config = ExperimentConfig.from_yaml(REPO_ROOT / "configs" / "heterogeneous_routes.yaml")
RISK_EPISODES = 5 if SMOKE else (450 if QUICK else full_risk_config.episodes)
RISK_SEEDS = (0,) if SMOKE else (tuple(range(5)) if QUICK else full_risk_config.seeds)
risk_agents = tuple(
    replace(
        agent,
        parameters={
            **agent.parameters,
            "epsilon": {
                "kind": "linear",
                "start": 0.30,
                "end": 0.03,
                "duration": max(40, RISK_EPISODES * 20),
            },
        },
    )
    for agent in full_risk_config.agents
)
risk_config = replace(
    full_risk_config,
    episodes=RISK_EPISODES,
    seeds=RISK_SEEDS,
    agents=risk_agents,
    snapshot_interval=2 if SMOKE else (25 if QUICK else full_risk_config.snapshot_interval),
    policy_evaluation=replace(
        full_risk_config.policy_evaluation,
        interval_episodes=2 if SMOKE else (75 if QUICK else 250),
        episodes_per_checkpoint=1 if SMOKE else (5 if QUICK else 10),
    ),
    execution=ExecutionSpec(parallel_workers=1 if QUICK or SMOKE else 4),
    artifacts=replace(
        full_risk_config.artifacts,
        output_dir=RESULTS_DIR,
        flush_rows=200 if SMOKE else 10_000,
    ),
)
risk_result = run_or_reopen("risk", risk_config)
risk_training = add_method(risk_result.training_episodes)
risk_evaluations = add_method(evaluation_checkpoint_summary(risk_result.evaluations))
assert set(risk_training["agent"]) == set(AGENT_ORDER)


In [ ]:
risk_env = make_environment(risk_config.environments[0])
risk_exact = value_iteration(risk_env.exact_mdp(), gamma=0.98)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
plot_maze(risk_env, ax=axes[0], title="Risky shortcut and detour")
plot_transition_noise(risk_env, ax=axes[1], title="Intended-action reliability")
plot_policy(
    risk_exact.policy,
    risk_env,
    values=risk_exact.values,
    ax=axes[2],
    title="Exact optimal policy",
)
plt.tight_layout()
plt.show()


### Learning behavior versus deployed behavior

A method can look worse during training simply because its exploratory
actions are being scored. Conversely, a smooth training curve can conceal a
brittle greedy policy. The two panels below must be read together.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_learning_curves(
    risk_training,
    metric="episode_return",
    group="method",
    smooth=min(25, RISK_EPISODES),
    n_resamples=N_RESAMPLES,
    ax=axes[0],
)
plot_learning_curves(
    risk_evaluations,
    metric="episode_return",
    x="checkpoint_episode",
    group="method",
    n_resamples=N_RESAMPLES,
    ax=axes[1],
)
axes[0].set_title("Exploratory training return")
axes[1].set_title("Frozen greedy held-out return")
plt.tight_layout()
plt.show()

risk_final = last_checkpoint(risk_evaluations)
risk_final_plot = risk_final.rename(columns={"checkpoint_episode": "episode"})
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_final_distribution(
    risk_final_plot,
    metric="episode_return",
    group="method",
    last_episodes=1,
    ax=axes[0],
)
plot_final_distribution(
    risk_final_plot,
    metric="success",
    group="method",
    last_episodes=1,
    ax=axes[1],
)
axes[0].set_title("Final held-out return by training seed")
axes[1].set_title("Final held-out success by training seed")
plt.tight_layout()
plt.show()


In [ ]:
risk_contrasts = []
for comparison in AGENT_ORDER:
    if comparison == "q_learning":
        continue
    contrast = paired_seed_contrast(
        risk_final,
        metric="episode_return",
        factor="agent",
        baseline="q_learning",
        comparison=comparison,
        pair_by=("seed",),
        strata=("evaluation_scenario",),
        n_resamples=N_RESAMPLES,
        random_seed=17,
    )
    risk_contrasts.append(contrast.summary)
risk_contrast_summary = pd.concat(risk_contrasts, ignore_index=True)
risk_contrast_summary["label"] = risk_contrast_summary["comparison"].map(AGENT_LABELS)
display(
    risk_contrast_summary[
        ["label", "n_pairs", "mean_difference", "ci_low", "ci_high", "win_rate"]
    ]
)
plot_paired_contrasts(
    risk_contrast_summary,
    label="label",
    difference_label="Held-out return difference versus Q-learning",
    title="Matched-seed final contrasts",
)
plt.show()


### Convergence and policy stability

Exact error is informative here, with one caveat: while epsilon remains
nonzero, SARSA and Expected SARSA evaluate an exploratory behavior policy.
Their larger distance from greedy Q-star can therefore be an estimand
difference rather than a broken update.


In [ ]:
risk_snapshots = add_method(risk_result.snapshots.query("episode >= 0"))
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
plot_learning_curves(
    risk_snapshots,
    metric="q_error_inf",
    group="method",
    log_y=True,
    n_resamples=N_RESAMPLES,
    ax=axes[0],
)
plot_learning_curves(
    risk_snapshots,
    metric="policy_disagreement",
    group="method",
    n_resamples=N_RESAMPLES,
    ax=axes[1],
)
plot_final_distribution(
    risk_training,
    metric="td_error_variance",
    group="method",
    last_episodes=max(1, min(75, RISK_EPISODES // 3)),
    ax=axes[2],
)
axes[0].set_title("Sup-norm Q error")
axes[1].set_title("Tie-aware policy disagreement")
axes[2].set_title("Late within-episode TD variance")
plt.tight_layout()
plt.show()

risk_threshold = episodes_to_threshold(
    risk_snapshots,
    metric="policy_disagreement",
    threshold=0.10,
    sustain=2,
    groups=("trial_id", "agent"),
)
display(
    risk_threshold.groupby("agent", as_index=False)
    .agg(
        median_episodes=("episodes_to_threshold", "median"),
        fraction_reached=("reached", "mean"),
    )
    .assign(method=lambda frame: frame["agent"].map(AGENT_LABELS))
    [["method", "median_episodes", "fraction_reached"]]
)


In [ ]:
risk_final_rows, risk_q_tables = final_q_tables(risk_result)
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
plot_policy(risk_exact.policy, risk_env, ax=axes[0, 0], title="Exact optimal policy")
for ax, agent in zip(axes.flat[1:], AGENT_ORDER, strict=False):
    trial_ids = risk_final_rows.loc[risk_final_rows["agent"].eq(agent), "trial_id"]
    policy, consensus = modal_policy([risk_q_tables[trial_id] for trial_id in trial_ids])
    plot_policy(
        policy,
        risk_env,
        values=consensus,
        ax=ax,
        cmap="Blues",
        vmin=0.0,
        vmax=1.0,
        colorbar_label="fraction of seeds choosing modal action",
        title=AGENT_LABELS[agent],
    )
axes.flat[-1].set_visible(False)
plt.show()


**Act I reading rule.** Prefer the paired held-out contrast and the raw seed
distribution to a visual ranking of smoothed lines. The policy atlas is a
modal summary across seeds; its blue background reveals where that summary
is stable and where a single representative arrow would be misleading.


## Act II — Drift: the target moves

The environment now drops action reliability after a fixed number of
decisions. In the current maze API this clock resets at every episode, so
this is a repeated, event-aligned shock—not one surprise halfway through
training. That repetition is useful: every episode provides a before/after
trace, while training seeds remain the independent replicates.

No stationary Q-star exists for this act. Exact comparison is disabled and
every transition is retained because adjacency is essential. We align on
the logged action reliability actually used for each transition; the
reported next regime changes one row earlier than the affected dynamics.


In [ ]:
DRIFT_CHANGE_STEP = 8
DRIFT_EPISODES = 5 if SMOKE else (250 if QUICK else 1_500)
DRIFT_SEEDS = (0,) if SMOKE else (tuple(range(4)) if QUICK else tuple(range(20)))
drift_agents = tuple(
    AgentSpec(
        name=name,
        kind=name,
        parameters={
            "learning_rate": 0.12,
            "gamma": 0.98,
            "epsilon": {
                "kind": "linear",
                "start": 0.30,
                "end": 0.04,
                "duration": max(40, DRIFT_EPISODES * 25),
            },
        },
    )
    for name in AGENT_ORDER
)
drift_environment = EnvironmentSpec(
    name="repeated_reliability_shock",
    kind="stochastic_maze",
    parameters={
        "shape": (3, 15),
        "start": (1, 0),
        "goals": {(1, 14): 5.0},
        "blocked_cells": [(0, 5), (2, 9)],
        "action_reliability": 0.98,
        "slip_weights": {"left": 0.45, "right": 0.45, "stay": 0.10},
        "step_reward": -0.03,
        "max_episode_steps": 70,
        "nonstationarity": NonstationarityConfig(
            mode="abrupt",
            reliability_multipliers=(1.0, 0.55),
            reward_multipliers=(1.0, 1.0),
            change_step=DRIFT_CHANGE_STEP,
        ),
    },
)
drift_config = ExperimentConfig(
    name="policies_under_repeated_drift",
    episodes=DRIFT_EPISODES,
    seeds=DRIFT_SEEDS,
    environments=(drift_environment,),
    agents=drift_agents,
    snapshot_interval=2 if SMOKE else (25 if QUICK else 50),
    exact_reference=False,
    policy_evaluation=PolicyEvaluationSpec(
        enabled=True,
        interval_episodes=2 if SMOKE else (50 if QUICK else 150),
        episodes_per_checkpoint=1 if SMOKE else (4 if QUICK else 10),
        include_initial=True,
        include_final=True,
        scenarios=(EvaluationScenario(name="repeated_shock"),),
    ),
    execution=ExecutionSpec(parallel_workers=1 if QUICK or SMOKE else 4),
    artifacts=ArtifactSpec(
        output_dir=RESULTS_DIR,
        table_format="auto",
        flush_rows=200 if SMOKE else 10_000,
        save_q_snapshots=True,
        step_retention=StepRetentionSpec(mode="all"),
    ),
    tags={"question": "event_aligned_td_response_to_repeated_drift"},
)
drift_result = run_or_reopen("drift", drift_config)
drift_training = add_method(drift_result.training_episodes)
drift_evaluations = add_method(evaluation_checkpoint_summary(drift_result.evaluations))
drift_steps = add_method(drift_result.steps)
assert not drift_result.snapshots["exact_evaluation_available"].any()


In [ ]:
late_drift_steps = drift_steps.loc[
    drift_steps["episode"].ge(max(0, DRIFT_EPISODES // 2))
].copy()
late_drift_steps["absolute_td_error"] = late_drift_steps["td_error"].abs()
per_trial_step = (
    late_drift_steps.groupby(
        ["trial_id", "agent", "method", "seed", "step"],
        as_index=False,
    )
    .agg(
        mean_absolute_td_error=("absolute_td_error", "mean"),
        mean_reward=("reward", "mean"),
        action_reliability=("env_action_reliability", "mean"),
    )
)
drift_profile = (
    per_trial_step.groupby(["method", "step"], as_index=False)
    .agg(
        mean_absolute_td_error=("mean_absolute_td_error", "mean"),
        sem_absolute_td_error=("mean_absolute_td_error", "sem"),
        mean_reward=("mean_reward", "mean"),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for method, sample in drift_profile.groupby("method", sort=False):
    axes[0].plot(sample["step"], sample["mean_absolute_td_error"], label=method)
    axes[0].fill_between(
        sample["step"],
        sample["mean_absolute_td_error"] - sample["sem_absolute_td_error"].fillna(0),
        sample["mean_absolute_td_error"] + sample["sem_absolute_td_error"].fillna(0),
        alpha=0.15,
    )
    axes[1].plot(sample["step"], sample["mean_reward"], label=method)
for ax in axes:
    ax.axvline(DRIFT_CHANGE_STEP, color="black", linestyle="--", label="reliability drop")
    ax.grid(alpha=0.22)
    ax.legend(frameon=False)
    ax.set_xlabel("step within episode")
axes[0].set(ylabel="mean absolute TD error", title="Event-aligned TD shock")
axes[1].set(ylabel="mean reward", title="Reward around the same shock")
plt.tight_layout()
plt.show()

late_drift_steps["window"] = np.select(
    [
        late_drift_steps["step"].lt(DRIFT_CHANGE_STEP),
        late_drift_steps["step"].lt(DRIFT_CHANGE_STEP + 5),
    ],
    ["before", "shock"],
    default="later",
)
drift_windows = (
    late_drift_steps.groupby(
        ["trial_id", "agent", "method", "seed", "window"],
        as_index=False,
    )
    .agg(
        mean_absolute_td_error=("absolute_td_error", "mean"),
        mean_reward=("reward", "mean"),
    )
)
display(
    drift_windows.groupby(["method", "window"], as_index=False)
    .agg(
        mean_absolute_td_error=("mean_absolute_td_error", "mean"),
        mean_reward=("mean_reward", "mean"),
        n_trials=("trial_id", "nunique"),
    )
)

plot_learning_curves(
    drift_evaluations,
    metric="episode_return",
    x="checkpoint_episode",
    group="method",
    n_resamples=N_RESAMPLES,
)
plt.title("Greedy performance in the repeated-shock environment")
plt.show()


**Act II reading rule.** A TD spike demonstrates surprise under the current
estimates; it is not by itself a change-point detector or proof of
adaptation. Recovery claims need the later window and held-out return.
Because the clock restarts, these results do not describe one irreversible
lifetime regime switch.


## Act III — Memory: when position is not a state

One wall follows a two-state Markov chain. We vary persistence while holding
its stationary probability of being present at one half:

$$
\pi_{\mathrm{present}}=\frac{p_{01}}{p_{01}+1-p_{11}},
\qquad \rho=p_{11}-p_{01}.
$$

Position-only Q-learning and wall-aware Q-learning receive the same seeds
and budgets. The latter observes the position plus the current wall bit and
therefore indexes the exact augmented MDP. This is a representation control,
not a new backup-rule tournament. If the wall bit changes the best action,
no position-only Q table can express both conditional policies.


In [ ]:
MEMORY_EPISODES = 5 if SMOKE else (450 if QUICK else 2_500)
MEMORY_SEEDS = (0,) if SMOKE else (tuple(range(5)) if QUICK else tuple(range(20)))
WALL_EDGE = ((1, 2), (1, 3))
PERSISTENCE = (
    (0.0, 0.50, 0.50),
    (0.6, 0.20, 0.80),
    (0.9, 0.05, 0.95),
)
memory_environments = []
memory_metadata = {}
for rho, p01, p11 in PERSISTENCE:
    parameters = {
        "shape": (3, 5),
        "start": (1, 0),
        "goals": {(1, 4): 4.0},
        "markov_walls": [
            MarkovWall(
                edge=WALL_EDGE,
                p01=p01,
                p11=p11,
                initial_probability=0.5,
            )
        ],
        "action_reliability": 0.98,
        "slip_weights": {"left": 0.45, "right": 0.45, "stay": 0.10},
        "step_reward": -0.03,
        "max_episode_steps": 60,
    }
    suffix = str(rho).replace(".", "_")
    for representation, kind in (
        ("position only", "stochastic_maze"),
        ("position + wall", "stochastic_maze_wall_state"),
    ):
        name = f"{representation.replace(' ', '_').replace('+', 'plus')}_rho_{suffix}"
        memory_environments.append(
            EnvironmentSpec(name=name, kind=kind, parameters=parameters)
        )
        memory_metadata[name] = (representation, rho)

memory_agent = AgentSpec(
    name="q_learning",
    kind="q_learning",
    parameters={
        "learning_rate": 0.12,
        "gamma": 0.98,
        "epsilon": {
            "kind": "linear",
            "start": 0.30,
            "end": 0.03,
            "duration": max(40, MEMORY_EPISODES * 18),
        },
    },
)
memory_config = ExperimentConfig(
    name="markov_wall_memory",
    episodes=MEMORY_EPISODES,
    seeds=MEMORY_SEEDS,
    environments=tuple(memory_environments),
    agents=(memory_agent,),
    snapshot_interval=2 if SMOKE else (25 if QUICK else 50),
    exact_reference=True,
    policy_evaluation=PolicyEvaluationSpec(
        enabled=True,
        interval_episodes=2 if SMOKE else (75 if QUICK else 250),
        episodes_per_checkpoint=1 if SMOKE else (5 if QUICK else 10),
        include_initial=True,
        include_final=True,
        scenarios=(EvaluationScenario(name="matched_markov_wall"),),
    ),
    execution=ExecutionSpec(parallel_workers=1 if QUICK or SMOKE else 4),
    artifacts=ArtifactSpec(
        output_dir=RESULTS_DIR,
        table_format="auto",
        flush_rows=200 if SMOKE else 10_000,
        save_q_snapshots=True,
        step_retention=StepRetentionSpec(
            mode="all" if SMOKE else "sample",
            fraction=1.0 if SMOKE else (0.20 if QUICK else 0.05),
            keep_terminal=True,
            keep_events=True,
        ),
    ),
    tags={"question": "state_representation_under_markov_wall_memory"},
)
memory_result = run_or_reopen("memory", memory_config)

def add_memory_factors(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["representation"] = result["environment"].map(
        lambda name: memory_metadata[name][0]
    )
    result["rho"] = result["environment"].map(lambda name: memory_metadata[name][1])
    return result

memory_evaluations = add_memory_factors(
    evaluation_checkpoint_summary(memory_result.evaluations)
)
memory_final = last_checkpoint(memory_evaluations)
memory_snapshots = add_memory_factors(memory_result.snapshots.query("episode >= 0"))
exact_availability = (
    memory_snapshots.groupby(["representation", "rho"], as_index=False)[
        "exact_evaluation_available"
    ].all()
)
display(exact_availability)
assert not exact_availability.query("representation == 'position only'")[
    "exact_evaluation_available"
].any()
assert exact_availability.query("representation == 'position + wall'")[
    "exact_evaluation_available"
].all()


In [ ]:
fig, axes = plt.subplots(1, len(PERSISTENCE), figsize=(16, 4.4), sharey=True)
for ax, (rho, _, _) in zip(axes, PERSISTENCE, strict=True):
    sample = memory_final.loc[memory_final["rho"].eq(rho)].rename(
        columns={"checkpoint_episode": "episode"}
    )
    plot_final_distribution(
        sample,
        metric="episode_return",
        group="representation",
        last_episodes=1,
        ax=ax,
    )
    ax.set_title(rf"$\rho={rho:g}$")
fig.suptitle("Held-out return as wall memory increases")
plt.tight_layout()
plt.show()

memory_contrast = paired_seed_contrast(
    memory_final,
    metric="episode_return",
    factor="representation",
    baseline="position only",
    comparison="position + wall",
    pair_by=("seed",),
    strata=("rho", "evaluation_scenario"),
    n_resamples=N_RESAMPLES,
    random_seed=23,
)
memory_contrast_summary = memory_contrast.summary.copy()
memory_contrast_summary["label"] = memory_contrast_summary["rho"].map(
    lambda rho: rf"wall-aware minus position-only, $\rho={rho:g}$"
)
display(
    memory_contrast_summary[
        ["rho", "n_pairs", "mean_difference", "ci_low", "ci_high", "win_rate"]
    ]
)
plot_paired_contrasts(
    memory_contrast_summary,
    label="label",
    difference_label="Held-out return difference",
    title="Value of observing the wall bit",
)
plt.show()

plot_learning_curves(
    memory_snapshots.loc[memory_snapshots["representation"].eq("position + wall")],
    metric="q_error_inf",
    group="rho",
    log_y=True,
    n_resamples=N_RESAMPLES,
)
plt.title("Wall-aware Q-learning versus its augmented Q-star")
plt.show()


In [ ]:
target_rho = max(rho for rho, _, _ in PERSISTENCE)
target_specs = [
    spec
    for spec in memory_config.environments
    if memory_metadata[spec.name][1] == target_rho
]
position_spec = next(
    spec for spec in target_specs if memory_metadata[spec.name][0] == "position only"
)
wall_spec = next(
    spec for spec in target_specs if memory_metadata[spec.name][0] == "position + wall"
)
position_env = make_environment(position_spec)
wall_env = make_environment(wall_spec)
augmented_exact = value_iteration(wall_env.exact_mdp(), gamma=0.98)
base_env = wall_env.unwrapped
open_indices = np.arange(base_env.n_states) * 2
closed_indices = open_indices + 1

memory_final_rows, memory_q_tables = final_q_tables(memory_result)
position_row = memory_final_rows.loc[
    memory_final_rows["environment"].eq(position_spec.name)
    & memory_final_rows["seed"].eq(POLICY_SEED)
].iloc[0]
wall_row = memory_final_rows.loc[
    memory_final_rows["environment"].eq(wall_spec.name)
    & memory_final_rows["seed"].eq(POLICY_SEED)
].iloc[0]
position_q = memory_q_tables[position_row["trial_id"]]
wall_q = memory_q_tables[wall_row["trial_id"]]

fig, axes = plt.subplots(2, 3, figsize=(16, 8.5), constrained_layout=True)
plot_policy(
    augmented_exact.policy[open_indices],
    base_env,
    values=augmented_exact.values[open_indices],
    ax=axes[0, 0],
    title="Exact: wall absent",
)
plot_policy(
    augmented_exact.policy[closed_indices],
    base_env,
    values=augmented_exact.values[closed_indices],
    ax=axes[0, 1],
    title="Exact: wall present",
)
plot_policy(
    np.argmax(position_q, axis=1),
    position_env,
    values=np.max(position_q, axis=1),
    ax=axes[0, 2],
    title=f"Position only, seed {POLICY_SEED}",
)
plot_policy(
    np.argmax(wall_q[open_indices], axis=1),
    base_env,
    values=np.max(wall_q[open_indices], axis=1),
    ax=axes[1, 0],
    title="Wall-aware learned: absent",
)
plot_policy(
    np.argmax(wall_q[closed_indices], axis=1),
    base_env,
    values=np.max(wall_q[closed_indices], axis=1),
    ax=axes[1, 1],
    title="Wall-aware learned: present",
)
conflict = augmented_exact.policy[open_indices] != augmented_exact.policy[closed_indices]
plot_policy(
    augmented_exact.policy[open_indices],
    base_env,
    values=conflict.astype(float),
    ax=axes[1, 2],
    cmap="Reds",
    vmin=0.0,
    vmax=1.0,
    colorbar_label="optimal action depends on wall",
    title="Where memory changes the action",
)
plt.show()


In [ ]:
memory_steps = add_memory_factors(memory_result.steps)
decision_state = position_env.state_to_index[(1, 2)]
wall_conditioned_td = memory_steps.loc[
    memory_steps["representation"].eq("position only")
    & memory_steps["rho"].eq(target_rho)
    & memory_steps["state"].eq(decision_state)
].copy()
if wall_conditioned_td.empty:
    print("No retained decision-state rows in this small run.")
else:
    wall_conditioned_td["wall"] = wall_conditioned_td["env_decision_wall_mask"].map(
        {0: "absent", 1: "present"}
    )
    per_trial_wall = (
        wall_conditioned_td.groupby(["trial_id", "seed", "wall"], as_index=False)
        .agg(
            mean_absolute_td_error=("absolute_td_error", "mean"),
            visits=("td_error", "size"),
        )
    )
    display(per_trial_wall)
    samples = [
        per_trial_wall.loc[
            per_trial_wall["wall"].eq(label), "mean_absolute_td_error"
        ].to_numpy()
        for label in ("absent", "present")
    ]
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.boxplot(samples, tick_labels=["wall absent", "wall present"], showfliers=False)
    ax.set(
        ylabel="per-seed mean absolute TD error",
        title="Position-only ambiguity at the blocked edge",
    )
    ax.grid(axis="y", alpha=0.22)
    plt.show()


## Synthesis

Better backups help when the problem is estimation. They cannot manufacture
stationarity or restore information omitted from the state.

- Act I is the clean algorithm comparison: stationary, fully observed, exact,
  paired, and evaluated without updates.
- Act II measures response to a repeated target shift. It supports claims
  about event-aligned surprise and recovery, not a one-time lifetime change.
- Act III changes temporal memory without changing average wall frequency.
  The gap between position-only and wall-aware learning is the value of
  state information, not evidence that one backup rule is universally best.

QUICK mode validates these mechanics. For stable confidence intervals, use
the full risk YAML from the command line and set QUICK to False for the two
targeted notebook runs. Reopen the resulting directories through
EXISTING_RUNS whenever you want to iterate on the analysis.
